In [1]:
import json, pandas as pd
from pathlib import Path
from IPython.display import display, HTML

PROJECT_ROOT = Path.cwd().parent
RESULTS_DIR = PROJECT_ROOT / "data" / "results"

RUN_NAME = "run_fin_2step_pot_gemma-4-26b-a4b_top10_20260430_103224"
jsonl_path = RESULTS_DIR / f"{RUN_NAME}.jsonl"
sidecar_path = RESULTS_DIR / f"{RUN_NAME}.json"

# JSONL: hat q_uid, gt_answers, lenient_match, retrieval_hit
with open(jsonl_path, encoding="utf-8") as f:
    rows = [json.loads(line) for line in f]

# Sidecar: hat generated_code
with open(sidecar_path, encoding="utf-8") as f:
    sidecar = json.load(f)

# Index sidecar by (question, doc_name) for join
code_lookup = {(r["question"], r["doc_name"]): r["generated_code"] for r in sidecar}

# Fehler filtern: retrieval_hit=True aber lenient_match=False
errors = []
for r in rows:
    if r["retrieval_hit"] and not r["lenient_match"]:
        errors.append({
            "quid": r["q_uid"],
            "question": r["question"],
            "extracted_facts": r["extracted_facts"],
            "generated_code": code_lookup.get((r["question"], r["doc_name"]), ""),
            "llm_answer": r.get("code_output") or r.get("display_answer", ""),
            "gt_answer": ", ".join(str(a) for a in r["gt_answers"]),
        })

df = pd.DataFrame(errors)
print(f"{len(df)} Fehler (retrieval hit, wrong answer)")
df.head(3)

183 Fehler (retrieval hit, wrong answer)


,quid,question,extracted_facts,generated_code,llm_answer,gt_answer
0,AAL/2017/page_10.pdf-3,what is the percentage change in the average p...,FACTS:\n- Increase in average price per gallon...,# The extracted facts state that there was an ...,21.4,"21.8%, 0.21831"
1,AAL/2017/page_10.pdf-4,what are the total operating expenses based on...,FACTS:\n- Mainline and regional aircraft fuel ...,# Extracted facts\nfuel_expense_2017 = 7510 #...,5.08,"38121.8, 38121.82741"
2,AAPL/2002/page_63.pdf-2,what percentage of total minimum lease payment...,FACTS:\n- Minimum lease payments in 2004 : 78 ...,# Given facts\nminimum_lease_payments_2004 = 7...,16.81,"17%, 0.1681"


In [ ]:
# Tabelle als HTML-Datei speichern
out_path = RESULTS_DIR / f"{RUN_NAME}_error_comparison.html"

html_rows = []
for _, r in df.iterrows():
    facts = (r['extracted_facts'] or "").replace("\n", "<br>")
    code = (r['generated_code'] or "").replace("\n", "<br>")
    html_rows.append(f"""<tr>
        <td>{r['quid']}</td>
        <td>{r['question']}</td>
        <td><pre style='white-space:pre-wrap'>{r['extracted_facts'] or ''}</pre></td>
        <td><pre style='white-space:pre-wrap'>{r['generated_code'] or ''}</pre></td>
        <td style='font-weight:bold'>{r['llm_answer']}</td>
        <td style='font-weight:bold;color:green'>{r['gt_answer']}</td>
    </tr>""")

html = f"""<html><head><meta charset='utf-8'>
<style>
table {{ border-collapse:collapse; width:100%; font-family:monospace; font-size:12px; }}
th, td {{ border:1px solid #ccc; padding:6px; vertical-align:top; }}
th {{ background:#f0f0f0; position:sticky; top:0; }}
pre {{ margin:0; max-height:200px; overflow:auto; font-size:11px; }}
</style></head><body>
<h2>{len(df)} Fehler (retrieval hit, wrong answer)</h2>
<table>
<tr><th>quid</th><th>question</th><th>extracted_facts</th><th>generated_code</th><th>llm_answer</th><th>gt_answer</th></tr>
{''.join(html_rows)}
</table></body></html>"""

with open(out_path, "w", encoding="utf-8") as f:
    f.write(html)

print(f"Gespeichert: {out_path}")